In [ ]:
from IPython.display import clear_output

import pandas as pd
import numpy as np
import lightgbm as lgb
import seaborn as sns

from featurewiz import FeatureWiz
from matplotlib import pyplot as plt
from openfe import OpenFE, transform
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report #accuracy_score, precision_score, recall_score, 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
seed = 42
y_col = "Class"

## 1.データの読み込み

In [ ]:
df_data = pd.read_csv("./data/data.csv")

In [ ]:
df_data.head(3)

## 2. ベースライン

In [ ]:
def make_dataset(df, y_col):

    X = df.drop(y_col, axis=1).copy()
    y = df[y_col].copy()

    # テストデータを全体の20%にする
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    # 検証データを全体の20%にする
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=seed
    )

    return (
        X_train.reset_index(drop=True), X_val.reset_index(drop=True), X_test.reset_index(drop=True),
        y_train.reset_index(drop=True), y_val.reset_index(drop=True), y_test.reset_index(drop=True)
    )

In [ ]:
def train_predict_eval(
    model,
    X_train, y_train, 
    X_val, y_val, 
    X_test, y_test,
):

    print("train starts")
    
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    print("prediction starts")
    y_pred = model.predict(X_test)

    print("pred ends\n")

    dict_report = classification_report(y_test, y_pred, output_dict=True)

    return pd.DataFrame(dict_report).T


In [ ]:
model_baseline = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=df_data[y_col].nunique(),
    class_weight="balanced",
    random_state=seed,
    verbose=-1,        # ログを非表示
    n_estimators=1000  # Early Stoppingを使うため大きめの数値を設定
)

In [ ]:
X_train_baseline, X_val_baseline, X_test_baseline, y_train_baseline, y_val_baseline, y_test_baseline = make_dataset(
    df=df_data,
    y_col=y_col
)

In [ ]:
df_report_baseline = train_predict_eval(
    model=model_baseline,
    X_train=X_train_baseline, y_train=y_train_baseline, 
    X_val=X_val_baseline, y_val=y_val_baseline, 
    X_test=X_test_baseline, y_test=y_test_baseline,
)

In [ ]:
df_report_baseline

## 3. 特徴量生成・選択・次元削減手法の比較

### 3.1 baseline + OpenFE

#### 3.1.1　OpenFEで特徴量生成

In [ ]:
# OpenFEのインスタンス
ofe = OpenFE()

In [ ]:
# ベースラインの学習データを基準に、新たな特徴量の計算

ofe_features = ofe.fit(
    data=X_train_baseline.copy(), # .copy()がないと元のdfが書き換わる
    label=y_train_baseline,
    n_jobs=1,
    stage2_params={"verbose": -1} # openfe内部のツリーモデルのログの非表示設定
) 

# かなりの量のログが出るので、notebook上のアウトプットを削除

clear_output(wait=True)
print("OpenFEのfit完了")

In [ ]:
# trainとvalを変換
X_train_ofe, X_val_ofe = transform(
    X_train_baseline, 
    X_val_baseline, 
    ofe_features,
    n_jobs=1
)

# trainとtestを変換
_, X_test_ofe = transform(
    X_train_baseline,  # 統計量を第1引数のデータ（学習データ）から計算する仕様のため、再度渡す
    X_test_baseline, 
    ofe_features,
    n_jobs=1
)


#### 3.1.2 学習と評価

In [ ]:
model_ofe = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=df_data[y_col].nunique(),
    class_weight="balanced",
    random_state=seed,
    verbose=-1,        # ログを非表示
    n_estimators=1000  # Early Stoppingを使うため大きめの数値を設定
)

In [ ]:
df_report_ofe = train_predict_eval(
    model=model_ofe,
    X_train=X_train_ofe, y_train=y_train_baseline, 
    X_val=X_val_ofe, y_val=y_val_baseline, 
    X_test=X_test_ofe, y_test=y_test_baseline,
)

In [ ]:
df_report_ofe

In [ ]:
# ベースラインとの比較
(df_report_ofe > df_report_baseline)

### 3.2 featurewiz

#### 3.2.1 特徴量選定

In [ ]:
# インスタンス作成
fw = FeatureWiz(corr_limit=0.70, verbose=2)

In [ ]:
X_train_baseline

In [ ]:
X_train_fw, _ = fw.fit_transform(
    X_train_baseline.copy(),
    y_train_baseline
)

In [ ]:
X_val_fw = fw.transform(X_val_baseline)
X_test_fw = fw.transform(X_test_baseline)


#### 3.2.2 学習と評価

In [ ]:
model_fw = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=df_data[y_col].nunique(),
    class_weight="balanced",
    random_state=seed,
    verbose=-1,        # ログを非表示
    n_estimators=1000  # Early Stoppingを使うため大きめの数値を設定
)

In [ ]:
df_report_fw = train_predict_eval(
    model=model_fw,
    X_train=X_train_fw, y_train=y_train_baseline, 
    X_val=X_val_fw, y_val=y_val_baseline, 
    X_test=X_test_fw, y_test=y_test_baseline,
)

In [ ]:
df_report_fw

In [ ]:
# ベースラインとの比較
(df_report_fw > df_report_baseline)

### 3.3 PCA

#### 3.3.1 次元削減

In [ ]:
# 標準化->PCAのパイプラインの作成

pca_pipe = make_pipeline(
    StandardScaler(),
    PCA(n_components="mle") #次元数は最尤推定で決定
)


In [ ]:
X_train_pca = pca_pipe.fit_transform(X_train_baseline)

In [ ]:
# 次元数の確認
X_train_pca.shape

In [ ]:
X_val_pca = pca_pipe.transform(X_val_baseline)

X_test_pca = pca_pipe.transform(X_test_baseline)

#### 3.3.2 学習と評価

In [ ]:
model_pca = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=df_data[y_col].nunique(),
    class_weight="balanced",
    random_state=seed,
    verbose=-1,        # ログを非表示
    n_estimators=1000  # Early Stoppingを使うため大きめの数値を設定
)

In [ ]:
df_report_pca = train_predict_eval(
    model=model_pca,
    X_train=X_train_pca, y_train=y_train_baseline, 
    X_val=X_val_pca, y_val=y_val_baseline, 
    X_test=X_test_pca, y_test=y_test_baseline,
)

In [ ]:
df_report_pca

In [ ]:
# ベースラインとの比較
(df_report_pca > df_report_baseline)

## 4. ベースライン+OpenFEにfeaturewiz or PCAを適応

In [ ]:
# ベースラインの特徴量の数を確認
X_train_baseline.shape

# →16個


In [ ]:
# OpenFEで増やした特徴量の数を確認
X_train_ofe

# →434個

### 4.1 featurewizで特徴量選定

#### 4.1.1 特徴量選定

In [ ]:
# インスタンス作成
fw_ofe = FeatureWiz(corr_limit=0.70, verbose=2)


In [ ]:
X_train_ofe_fw, _ = fw_ofe.fit_transform(
    X_train_ofe.copy(),
    y_train_baseline
)

In [ ]:
X_train_ofe_fw.shape

In [ ]:
X_val_ofe_fw = fw_ofe.transform(X_val_ofe.copy())
X_test_ofe_fw = fw_ofe.transform(X_test_ofe.copy())

#### 4.1.2 学習と評価

In [ ]:
model_ofe_fw = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=df_data[y_col].nunique(),
    class_weight="balanced",
    random_state=seed,
    verbose=-1,        # ログを非表示
    n_estimators=1000  # Early Stoppingを使うため大きめの数値を設定
)

In [ ]:
df_report_ofe_fw = train_predict_eval(
    model=model_ofe_fw,
    X_train=X_train_ofe_fw, y_train=y_train_baseline, 
    X_val=X_val_ofe_fw, y_val=y_val_baseline, 
    X_test=X_test_ofe_fw, y_test=y_test_baseline,
)

In [ ]:
df_report_ofe_fw

In [ ]:
# ベースラインとの比較
(df_report_ofe_fw > df_report_baseline)

In [ ]:
# OpenFEの有無で比較
(df_report_ofe_fw > df_report_fw)

### 4.2 PCAで次元圧縮

#### 4.2.1 次元圧縮

In [ ]:
# 学習データでfit
X_train_ofe_pca = pca_pipe.fit_transform(X_train_ofe.copy())


In [ ]:
# 何次元になったか、確認
X_train_ofe_pca.shape

In [ ]:
X_val_ofe_pca = pca_pipe.transform(X_val_ofe.copy())
X_test_ofe_pca = pca_pipe.transform(X_test_ofe.copy())

#### 4.2.2 学習と評価

In [ ]:
model_ofe_pca = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=df_data[y_col].nunique(),
    class_weight="balanced",
    random_state=seed,
    verbose=-1,        # ログを非表示
    n_estimators=1000  # Early Stoppingを使うため大きめの数値を設定
)


In [ ]:
df_report_ofe_pca = train_predict_eval(
    model=model_ofe_pca,
    X_train=X_train_ofe_pca, y_train=y_train_baseline, 
    X_val=X_val_ofe_pca, y_val=y_val_baseline, 
    X_test=X_test_ofe_pca, y_test=y_test_baseline,
)


In [ ]:
df_report_ofe_pca

In [ ]:
# ベースラインとの比較
(df_report_ofe_pca > df_report_baseline)

In [ ]:
# OpenFEの有無で比較
(df_report_ofe_pca > df_report_pca)

## 5.　精度の比較

In [ ]:
df_result = pd.DataFrame({
    "Method": [
        "Baseline",
        "OpenFE",
        "featurewiz",
        "PCA",
        "OpenFE + featurewiz",
        "OpenFE + PCA",
    ],
    "Features": [
        16,
        434,
        4,
        15,
        17,
        214,
    ],
    "Macro F1": [
        df_report_baseline.loc["macro avg", "f1-score"],
        df_report_ofe.loc["macro avg", "f1-score"],
        df_report_fw.loc["macro avg", "f1-score"],
        df_report_pca.loc["macro avg", "f1-score"],
        df_report_ofe_fw.loc["macro avg", "f1-score"],
        df_report_ofe_pca.loc["macro avg", "f1-score"],
    ]
})

In [ ]:
df_result

In [ ]:
df_result.plot.bar(
    x="Method",
    y="Macro F1",
    legend=False
)

plt.title("Comparison of Macro F1")
plt.ylabel("Macro F1")
plt.ylim(0.88, 0.95)
plt.xticks(rotation=45)
plt.show()

In [ ]:
ax = df_result.plot.scatter(
    x="Features",
    y="Macro F1"
)

for _, row in df_result.iterrows():
    ax.annotate(
        row["Method"],
        (row["Features"], row["Macro F1"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.title("Feature Count vs Macro F1")
plt.xlabel("Number of features")
plt.ylabel("Macro F1")
plt.show()